In [1]:
import langchain_openai

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [3]:
import pandas as pd

In [4]:
from langchain.docstore.document import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.retrievers import BM25Retriever
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA

In [5]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o-mini")


In [6]:
def load_excel_documents(folder_path):
    """
    Verilen klasör altındaki .xlsx veya .xls dosyalarını okur,
    satır bazında Document nesneleri oluşturur.
    """
    docs = []
    for file_name in os.listdir(folder_path):
        # Excel uzantısını kontrol et
        if file_name.lower().endswith(".xlsx") or file_name.lower().endswith(".xls"):
            full_path = os.path.join(folder_path, file_name)
            
            # Excel'i DataFrame olarak yükle
            df = pd.read_excel(full_path)

            # Her satırı bir Document olarak ekle (isterseniz hücre birleştirme vb. özelleştirebilirsiniz)
            for index, row in df.iterrows():
                # Tüm hücreleri tek bir metin halinde birleştiriyoruz
                row_text = " | ".join([f"{col}: {row[col]}" 
                                       for col in df.columns 
                                       if not pd.isnull(row[col])])

                # Boş satır gibi durumlar oluşmasın diye kontrol
                if row_text.strip():
                    doc = Document(
                        page_content=row_text,
                        metadata={"source": file_name, "row_index": index}
                    )
                    docs.append(doc)
    return docs

In [7]:
def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    """
    Doküman listesini belirtilen boyutta parçalara ayırarak geri döndürür.
    Metin uzun olursa chunk'a bölmek, BM25 arama kalitesini artırabilir.
    """
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    splitted_docs = []
    for doc in documents:
        chunks = text_splitter.split_text(doc.page_content)
        for chunk in chunks:
            splitted_docs.append(Document(
                page_content=chunk,
                metadata=doc.metadata
            ))
    return splitted_docs


In [8]:
def build_bm25_retriever(docs):
    """
    BM25Retriever oluşturur (kelime/frekans tabanlı).
    """
    retriever = BM25Retriever.from_documents(docs)
    return retriever

In [9]:
def create_qa_chain(retriever):
    """
    BM25Retriever kullanan bir RetrievalQA zinciri oluşturur.
    return_source_documents=True ile ilgili dökümanları da yanıtla birlikte alabiliriz.
    """
    llm = chatModel
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  # "stuff": bulunmuş metin parçalarını direkt birleştirir
        retriever=retriever,
        return_source_documents=True
    )
    return qa_chain

In [10]:
folder_path = "EXCEL"  # Excel dosyalarının bulunduğu klasör yolunu belirtin
print("Excel dosyaları yükleniyor...")
excel_docs = load_excel_documents(folder_path)
print(f"Yüklenen satır sayısı: {len(excel_docs)}")

Excel dosyaları yükleniyor...
Yüklenen satır sayısı: 583


In [11]:
# Metin uzun olursa parçalara ayırmak aranabilirliği artırır
splitted_docs = split_documents(excel_docs, chunk_size=1000, chunk_overlap=100)
print(f"Parçalanmış doküman sayısı: {len(splitted_docs)}")

print("BM25 retriever oluşturuluyor...")
retriever = build_bm25_retriever(splitted_docs)

print("RetrievalQA zinciri oluşturuluyor...")
qa_chain = create_qa_chain(retriever)

Parçalanmış doküman sayısı: 584
BM25 retriever oluşturuluyor...
RetrievalQA zinciri oluşturuluyor...


In [15]:
#query = "Soru:BIST 100-30 endeksinin yıl sonuna göre değişimi ne kadar"
query = "ankara ilinde 2013 yılında  ortalama eğitim süresi"

# Yeni LangChain sürümlerinde __call__ yerine invoke() öneriliyor.
result = qa_chain.invoke({"query": query})

answer = result["result"]
source_docs = result["source_documents"]

print("\nCevap:")
print(answer)

# Hangi Excel dosyasının hangi satırından geldiğini görmek için metadata:
print("\nKullanılan Kaynaklar:")
for i, doc in enumerate(source_docs, start=1):
    print(f"  {i}. {doc.metadata}")
print("-" * 50)


Cevap:
Bilmiyorum.

Kullanılan Kaynaklar:
  1. {'source': 'cinsiyete gore 25 yas ve uzeri nufusta egitim tamamlama orani (yuksekogretim).xls', 'row_index': 8}
  2. {'source': 'illere gore ortalama egitim suresi.xls', 'row_index': 2}
  3. {'source': 'illere gore ortalama egitim suresi.xls', 'row_index': 89}
  4. {'source': 'illere gore ortalama egitim suresi.xls', 'row_index': 90}
--------------------------------------------------
